In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import numpy as np
import torch.nn.functional as F
import torchvision
from torchvision.transforms import functional as TF
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
def load_image(path, target_height=None, target_width=None, device=None, invert=False):
    target_image = torchvision.io.read_image(path, mode=torchvision.io.ImageReadMode.RGB)
    target_image = target_image.float() / 255.0
    
    if invert:
        target_image = 1 - target_image
    
    if target_height is not None and target_width is not None:
        # 1. Get current dimensions
        _, h, w = target_image.shape
        
        # 2. Calculate the scaling factor to fit within the box
        # We use 'min' to ensure the image is fully contained
        scale = min(target_height / h, target_width / w)
        new_h, new_w = int(h * scale), int(w * scale)
        
        # 3. Resize with maintained aspect ratio
        target_image = TF.resize(target_image, [new_h, new_w], 
                                 antialias=True, 
                                 interpolation=torchvision.transforms.InterpolationMode.BICUBIC)
        
        # 4. Padding (Letterboxing) to reach the exact target_height/width
        pad_h = target_height - new_h
        pad_w = target_width - new_w
        
        # Distribute padding to center the image
        padding = [pad_w // 2, pad_h // 2, (pad_w + 1) // 2, (pad_h + 1) // 2]
        target_image = TF.pad(target_image, padding, fill=0) # fill with black

    if device is not None:
        target_image = target_image.to(device)

    return target_image
    
def plot_image(image):
    plt.figure(figsize=(6, 6))
    image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
    image_np = np.clip(image_np, 0, 1) # ensure values are in [0, 1]
    plt.imshow(image_np)
    #plt.axis("off")
    plt.show()

def plot_image_side_by_side(image, target, title=None):
    fig, (ax_gen, ax_target) = plt.subplots(1, 2, figsize=(12, 6))
    image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
    image_np = np.clip(image_np, 0, 1) # ensure values are in [0, 1]
    target_np = target.detach().cpu().numpy().transpose(1, 2, 0)
    target_np = np.clip(target_np, 0, 1) # ensure values are in [0, 1]

    ax_target.imshow(target_np)
    ax_target.set_title("Target Image")
    ax_gen.imshow(image_np)
    ax_gen.set_title("Generated")

    if title is not None:
        fig.suptitle(title)
    #plt.axis("off")
    plt.show()

def plot_image_live_side_by_side(gen_image, target_image, ax_gen=None, im_gen=None):
    # Convert AI image to numpy
    gen_np = gen_image.detach().cpu().numpy().transpose(1, 2, 0)
    gen_np = np.clip(gen_np, 0, 1)
    
    if ax_gen is None or im_gen is None:
        plt.ion()
        fig, (ax_gen, ax_target) = plt.subplots(1, 2, figsize=(12, 6))
        
        # 1. Target Plot (Static)
        target_np = target_image.detach().cpu().numpy().transpose(1, 2, 0)
        target_np = np.clip(target_np, 0, 1)
        ax_target.imshow(target_np)
        ax_target.set_title("Target Image")
        ax_target.axis("off")
        
        # 2. Generated Plot (Dynamic)
        im_gen = ax_gen.imshow(gen_np)
        ax_gen.set_title("Generated")
        ax_gen.axis("off")
            
        plt.show()
    else:
        fig = ax_gen.figure
        # Only update the AI image to keep it fast
        im_gen.set_data(gen_np)
            
    fig.canvas.draw_idle()
    plt.pause(0.01)
    
    return ax_gen, im_gen, fig

def print_param_summary(params, names=None):
    print("Parameter Summary:")
    print("="*50)
    total = 0
    for i, p in enumerate(params):
        name = names[i] if names is not None else f"param_{i}"
        shape = tuple(p.shape)
        count = p.numel()
        print(f"{name:10s} | shape: {str(shape):>15} | params: {count}")
        total += count
    print("="*50)
    print(f"Total params: {total}")
    print(f"Compression (H * W * 4) / total: {(H * W * 4)} / {total}: {(H * W * 4) / total :.2f}x")

In [ ]:
image_path = "pokemon.jpg"
invert = True
H, W = 32, 32
target_image = load_image(image_path, H, W, device, invert)
plot_image(target_image)

In [ ]:
wall = torch.zeros((3, H, W), device=device)

In [ ]:
N = 4
centers = (torch.rand(N, 2, device=device)).requires_grad_(True)
rads = (torch.rand(N, 1, device=device) * 0.1).requires_grad_(True)
colors = torch.rand(N, 3, device=device, requires_grad=True)
alphas = (torch.rand(N, device=device) * 0.8 + 0.1).requires_grad_(True)

In [ ]:
def render(canvas):
    _, H, W = canvas.shape
    N, _ = rads.shape
    
    # feature scaling
    grid_shape = torch.tensor([W, H], device=device)
    scaled_centers = centers * grid_shape
    scaled_rads = rads * min(W, H)

    # we cannot modify canvas in-place because we need the input canvas
    # for pytorch to later compute the gradient between old and new
    total_light = torch.zeros_like(canvas)

    for i in range(N):
        for x in range(W):
            for y in range(H):
                if (x-scaled_centers[i, 0])**2 + (y-scaled_centers[i, 1])**2 < scaled_rads[i]**2:
                    total_light[0, y, x] += colors[i, 0] * alphas[i]
                    total_light[1, y, x] += colors[i, 1] * alphas[i]
                    total_light[2, y, x] += colors[i, 2] * alphas[i]

    # returns a new tensor -> pytorch can calculate the gradient between old and new value
    return canvas + total_light

In [ ]:
output = render(wall)
plot_image_side_by_side(output, target_image, title=f"N={N}, H={H}, W={W}")

In [ ]:
params = [centers, rads, colors, alphas]
optimizer = torch.optim.Adam(params, lr=0.01)

print_param_summary(params, ["centers", "rads", "colors", "alphas"])
i = 0

In [ ]:
ax = None
im = None

try:
    loss = F.mse_loss
    last_loss = None
    while True:
        # Reset the gradients of all tensors
        optimizer.zero_grad()
        
        output = render(wall)
        loss = F.mse_loss(output, target_image)
        
        if last_loss is not None:
            delta = loss.item() - last_loss.item()
        else:
            delta = 0.0
        last_loss = loss
        
        loss.backward()
        optimizer.step()
        
        # UI Updates
        clear_output(wait=True)
        title = f"N={N} | Epoch: {i} | Loss: {loss.item():.6f} [Δ {delta:.6f}]"
        
        # Pass both the generated and target images
        ax, im, fig = plot_image_live_side_by_side(output, target_image, ax, im)
        fig.suptitle(title)
        display(fig)
        
        i += 1
except KeyboardInterrupt:
    pass